# e-SNLI — Gemma3-27b-it SAE Layer 31 · Density-Weighted Feature Steering

Workflow:
1. Load e-SNLI dataset, Gemma-3-27b-it model, and a residual-stream SAE at a configurable layer (default 31).
2. Randomly sample one example and build the NLI prompt.
3. Generate the model response and verify the `<label>` tag matches the ground-truth label.
4. Collect residual-stream activations at the SAE layer.
5. Encode with the SAE; collect every feature that fires on at least one token, together with per-token activation values and max activation.
6. Look up Neuronpedia descriptions and activation density (`frac_nonzero`) for all shortlisted features.
7. Attach `frac_nonzero` to the per-feature table.
8. Scale each feature's activations by `1 / max(frac_nonzero, ε)` — rare features are up-weighted.
9. Sort features by descending max scaled activation; break ties in favour of the higher feature index.
10–12. Steer each of the top-5 features negatively at `−10 × raw max activation`; record the steered response, steering value, and whether the correct label is preserved.

## Imports

In [1]:
import re
import sys
import os

import torch
import pandas as pd
from IPython.display import display

sys.path.insert(0, os.path.dirname(os.getcwd()))

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM

from src.configs import DatasetConfig, InferenceConfig, PromptStyle, SAEConfig
from src.dataset.esnli import ESNLI_Dataset
from src.SAE import JumpReLUSAE
from src.neuronpedia_client import NeuronpediaClient, build_sae_id

## Configuration

In [ ]:
LAYER = 31

sae_config = SAEConfig(
    repo_id="google/gemma-scope-2-27b-it",
    sae_type="resid_post",
    layer=LAYER,
    width="65k",
    l0="medium",
)

inference_config = InferenceConfig(
    max_new_tokens=512,
)

dataset_config = DatasetConfig(
    path="esnli/esnli",
    prompt_style=PromptStyle.CHAIN_OF_THOUGHT_TAGS,
    use_chat_template=False,
    hf_data_config={"split": "validation"},
    few_shot=False,
)

# Neuronpedia identifiers
NP_MODEL_ID = "gemma-3-27b-it"
NP_SAE_ID   = build_sae_id(sae_config)   # e.g. "31-gemmascope-2-res-65k"

# Small constant to avoid division by zero when frac_nonzero == 0
DENSITY_EPS = 1e-6

print(f"Model:       google/gemma-3-27b-it")
print(f"SAE layer:   {LAYER}")
print(f"SAE path:    {sae_config.sae_path}")
print(f"NP SAE ID:   {NP_SAE_ID}")

Model:       google/gemma-3-27b-it
SAE layer:   31
SAE path:    resid_post/layer_31_width_65k_l0_medium/params.safetensors
NP SAE ID:   31-gemmascope-2-res-65k


## HF Token

In [3]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

## Load Dataset

In [4]:
esnli_dataset = ESNLI_Dataset(dataset_config)
prompted_data = esnli_dataset.build_prompts()
esnli_df = prompted_data.to_pandas()
print(f"Dataset size: {len(esnli_df)} examples")
esnli_df.head(3)

Data successfully loaded.


Building prompts:   0%|          | 0/9842 [00:00<?, ? examples/s]

Dataset size: 9842 examples


,premise,hypothesis,label,explanation_1,explanation_2,explanation_3,gold_label,prompt
0,Two women are embracing while holding to go pa...,The sisters are hugging goodbye while holding ...,1,The to go packages may not be from lunch.,"Just because two women are embracing, does not...",Two women do not have to be sisters. Embracin...,neutral,<start_of_turn>user Task: Determine the logica...
1,Two women are embracing while holding to go pa...,Two woman are holding packages.,0,Saying the two women are holding packages is a...,Sentence 1 states that two women are holding t...,Women can embrace while they are holding packa...,entailment,<start_of_turn>user Task: Determine the logica...
2,Two women are embracing while holding to go pa...,The men are fighting outside a deli.,2,In the first sentence there is an action of af...,Women are different than men and embracing is ...,First sentence features two women and the seco...,contradiction,<start_of_turn>user Task: Determine the logica...


## Load Model + SAE

In [5]:
device    = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-27b-it")
model     = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-27b-it", device_map=device, dtype=torch.bfloat16
)
model.eval()
print("Model loaded.")

sae = JumpReLUSAE.from_pretrained(sae_config, device=device)
sae.eval()
d_model = sae.w_dec.shape[1]
d_sae   = sae.w_dec.shape[0]
print(f"SAE loaded. d_model={d_model}, d_sae={d_sae}")

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

Model loaded.
Load SAE resid_post/layer_31_width_65k_l0_medium/params.safetensors from google/gemma-scope-2-27b-it
SAE loaded. d_model=5376, d_sae=65536


## Helper Functions

In [ ]:
def generate_response(prompt: str):
    """Tokenize *prompt*, run greedy generation, return (full_text, full_ids, prompt_len)."""
    inputs     = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=inference_config.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    return text, output_ids, prompt_len


def collect_residual_activations(full_ids: torch.Tensor, layer_idx: int) -> torch.Tensor:
    """Hook the output of *layer_idx* to capture the residual stream.

    Returns a tensor of shape (n_tokens, d_model) on CPU.
    """
    cache  = {}
    layer  = model.model.language_model.layers[layer_idx]
    handle = layer.register_forward_hook(
        lambda _m, _i, out: cache.__setitem__(
            "resid",
            (out[0] if isinstance(out, tuple) else out).detach().squeeze(0),
        )
    )
    try:
        with torch.no_grad():
            model(input_ids=full_ids, use_cache=False)
    finally:
        handle.remove()
    return cache["resid"]   # (n_tokens, d_model)


def run_steered_generation(
    prompt_ids: torch.Tensor,
    layer_idx: int,
    steering_delta: torch.Tensor,
) -> torch.Tensor:
    """Run a single steered generation pass.

    Injects *steering_delta* (shape ``(d_model,)``) at every decoding step by
    hooking the output of the residual stream at *layer_idx*.

    Returns the full output token-ID tensor (1, seq_len).
    """
    layer = model.model.language_model.layers[layer_idx]

    def _hook(module, inp, out):
        hidden = out[0] if isinstance(out, tuple) else out
        hidden = hidden + steering_delta.to(dtype=hidden.dtype, device=hidden.device)
        if isinstance(out, tuple):
            return (hidden,) + out[1:]
        return hidden

    handle = layer.register_forward_hook(_hook)
    try:
        with torch.no_grad():
            steered_ids = model.generate(
                input_ids=prompt_ids,
                attention_mask=torch.ones_like(prompt_ids),
                max_new_tokens=inference_config.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
    finally:
        handle.remove()
    return steered_ids


def decode_response(ids: torch.Tensor) -> str:
    """Decode *ids* and return the model-turn text only."""
    full = tokenizer.decode(ids[0], skip_special_tokens=True)
    return full.split("<start_of_turn>model")[-1].strip()

## Sample + Prompt Generation

Randomly pick one example and verify the model produces the correct label. Re-sample up to `MAX_RETRIES` times if the model answers incorrectly.

In [12]:
SEED        = 42

sample_row  = None
sample_text = None
full_ids    = None
prompt_len  = None

candidate_df = esnli_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

for attempt, (_, row) in enumerate(candidate_df.iterrows()):
    prompt     = row["prompt"]
    gold_label = row["gold_label"].lower()

    text, out_ids, p_len = generate_response(prompt)
    predicted  = esnli_dataset.parse_model_answer(text).label

    print(f"  Predicted label: {predicted!r}  |  Gold label: {gold_label!r}")

    if predicted == gold_label:
        sample_row  = row
        sample_text = text
        full_ids    = out_ids
        prompt_len  = p_len
        print("Found a correctly-answered sample.")
        break

print(f"\nSelected sample index: {sample_row.name}")
print(f"Premise:    {sample_row['premise']}")
print(f"Hypothesis: {sample_row['hypothesis']}")
print(f"Gold label: {sample_row['gold_label']}")
print(f"Prompt tokens: {prompt_len}  |  Total tokens: {full_ids.shape[1]}")

  Predicted label: 'neutral'  |  Gold label: 'contradiction'
  Predicted label: 'neutral'  |  Gold label: 'contradiction'
  Predicted label: 'contradiction'  |  Gold label: 'contradiction'
Found a correctly-answered sample.

Selected sample index: 2
Premise:    A lady sitting on a bench that is against a building and under a poster of a man in a uniform waving.
Hypothesis: Nobody is sitting
Gold label: contradiction
Prompt tokens: 109  |  Total tokens: 186


## Collect Residual Stream Activations

Hook the layer output (= post-MLP residual stream) at `LAYER`. The resulting tensor has shape `(n_tokens, d_model)` and is the input expected by the residual-stream SAE.

In [13]:
resid_acts = collect_residual_activations(full_ids, LAYER)   # (n_tokens, d_model)
print(f"Residual activations shape: {resid_acts.shape}")

Residual activations shape: torch.Size([186, 5376])


## SAE Encoding + Feature Inventory

Encode the full-sequence residual activations with the SAE, then collect every feature that fires on **at least one token** across the whole sequence (prompt + generated). For each such feature, record:
- the token indices where it fires,
- its activation at every firing token,
- its maximum activation across all tokens.

In [14]:
with torch.no_grad():
    sae_acts = sae.encode(resid_acts.float())   # (n_tokens, d_sae)

print(f"SAE activations shape: {sae_acts.shape}")
print(f"Mean L0 over sequence: {(sae_acts > 0).float().sum(dim=-1).mean():.1f}")

# ── Features that fire at least once anywhere in the sequence ──────────────
fires_anywhere     = (sae_acts > 0).any(dim=0)                          # (d_sae,)
active_feature_ids = fires_anywhere.nonzero(as_tuple=False).squeeze(-1).tolist()
print(f"\nFeatures active on ≥1 token: {len(active_feature_ids)}")

# ── Per-feature inventory ──────────────────────────────────────────────────
# Pandas DataFrame: one row per active feature.
# `token_activations` (dict) and `firing_tokens` (list) are object-typed columns
# so we can cheaply attach Neuronpedia metadata and the scaled-activation columns later.
rows = []
for fi in active_feature_ids:
    token_act_vec  = sae_acts[:, fi].cpu()                               # (n_tokens,)
    firing_mask    = token_act_vec > 0
    firing_tokens  = firing_mask.nonzero(as_tuple=False).squeeze(-1).tolist()
    token_act_dict = {tok_idx: token_act_vec[tok_idx].item() for tok_idx in firing_tokens}
    max_act        = token_act_vec.max().item()
    rows.append({
        "feature_idx":       fi,
        "firing_tokens":     firing_tokens,     # list[int]: indices where feature fires
        "token_activations": token_act_dict,    # dict{token_idx: activation}
        "max_activation":    max_act,
    })

feature_df = pd.DataFrame(rows)
print(f"\nFeature DataFrame shape: {feature_df.shape}")
display(feature_df[["feature_idx", "max_activation", "firing_tokens"]].head(10))

SAE activations shape: torch.Size([186, 65536])
Mean L0 over sequence: 57.2

Features active on ≥1 token: 3812

Feature DataFrame shape: (3812, 4)


,feature_idx,max_activation,firing_tokens
0,0,3035.127197,"[5, 117, 130]"
1,1,1194.930664,"[0, 5, 35, 50, 117]"
2,6,20448.833984,[0]
3,8,1900.177734,[81]
4,12,16403.835938,[0]
5,14,214.954590,[9]
6,20,932.897461,"[96, 104, 123, 149, 167]"
7,25,1116.075806,"[13, 22, 24]"
8,26,586.263062,"[38, 78, 83, 124]"
9,29,408.593597,[76]


## Neuronpedia Lookup

Fetch the human-readable description and activation density (`frac_nonzero`) for every shortlisted feature.

In [15]:
client      = NeuronpediaClient(model_id=NP_MODEL_ID, sae_id=NP_SAE_ID)
np_features = client.get_features(feature_df["feature_idx"].tolist())

feature_df["description"]  = feature_df["feature_idx"].map(
    lambda fi: (np_features[fi].description  or "N/A") if fi in np_features else "N/A"
)
feature_df["frac_nonzero"] = feature_df["feature_idx"].map(
    lambda fi: (np_features[fi].frac_nonzero or 0.0)   if fi in np_features else 0.0
)

print(f"Fetched Neuronpedia data for {len(np_features)} features.")
display(feature_df[["feature_idx", "max_activation", "frac_nonzero", "description"]].head(10))

Fetched Neuronpedia data for 3812 features.


,feature_idx,max_activation,frac_nonzero,description
0,0,3035.127197,0.008827,key concepts and definitions related to signal...
1,1,1194.930664,0.032834,phrases that indicate self-awareness and accou...
2,6,20448.833984,0.000000,N/A
3,8,1900.177734,0.003397,content related to economic theories and discu...
4,12,16403.835938,0.000000,N/A
5,14,214.954590,0.005853,"formal and structured types of communication, ..."
6,20,932.897461,0.017307,information about blood tests and natural reme...
7,25,1116.075806,0.008481,references to established journalism practices...
8,26,586.263062,0.018897,the structure and content related to informati...
9,29,408.593597,0.000753,titles before names


## Density-Weighted Scaling

Scale each feature's activations by `1 / max(frac_nonzero, ε)`:
- a feature that fires rarely across the corpus (low `frac_nonzero`) gets *up-weighted*;
- a feature that fires on almost every token (high `frac_nonzero`) gets *down-weighted*.

In [16]:
def scale_by_density(activations: dict, frac_nonzero: float, eps: float = DENSITY_EPS) -> dict:
    """Return a new dict with each activation value divided by max(frac_nonzero, eps)."""
    weight = 1.0 / max(frac_nonzero, eps)
    return {tok: act * weight for tok, act in activations.items()}


feature_df["scaled_token_activations"] = feature_df.apply(
    lambda row: scale_by_density(row["token_activations"], row["frac_nonzero"]),
    axis=1,
)
feature_df["scaled_max_activation"] = feature_df["scaled_token_activations"].map(
    lambda d: max(d.values()) if d else 0.0
)

print("Scaling applied. Sample (raw vs scaled max activation):")
display(
    feature_df[["feature_idx", "frac_nonzero", "max_activation", "scaled_max_activation"]]
    .head(10)
)

Scaling applied. Sample (raw vs scaled max activation):


,feature_idx,frac_nonzero,max_activation,scaled_max_activation
0,0,0.008827,3035.127197,3.438402e+05
1,1,0.032834,1194.930664,3.639321e+04
2,6,0.000000,20448.833984,2.044883e+10
3,8,0.003397,1900.177734,5.593490e+05
4,12,0.000000,16403.835938,1.640384e+10
5,14,0.005853,214.954590,3.672345e+04
6,20,0.017307,932.897461,5.390328e+04
7,25,0.008481,1116.075806,1.315981e+05
8,26,0.018897,586.263062,3.102467e+04
9,29,0.000753,408.593597,5.429575e+05


## Sort Features

Sort descending by `scaled_max_activation`; break ties by preferring the *higher* feature index.

In [17]:
feature_df = (
    feature_df
    .sort_values(
        by=["scaled_max_activation", "feature_idx"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

print("Top-20 features by density-scaled max activation:")
pd.set_option("display.max_rows", 20)
display(
    feature_df[
        ["feature_idx", "frac_nonzero", "max_activation", "scaled_max_activation", "description"]
    ].head(20)
)

Top-20 features by density-scaled max activation:


,feature_idx,frac_nonzero,max_activation,scaled_max_activation,description
0,1686,0.000000e+00,24749.103516,2.474910e+10,N/A
1,1224,0.000000e+00,22984.554688,2.298455e+10,N/A
2,525,1.999982e-08,22778.083984,2.277808e+10,N/A
3,1107,9.999912e-09,22501.837891,2.250184e+10,impacts site selection
4,410,0.000000e+00,22484.263672,2.248426e+10,N/A
5,170,0.000000e+00,21948.062500,2.194806e+10,N/A
6,1723,0.000000e+00,21778.195312,2.177820e+10,N/A
7,1680,0.000000e+00,21651.904297,2.165190e+10,N/A
8,2028,0.000000e+00,21356.828125,2.135683e+10,N/A
9,1986,0.000000e+00,21299.931641,2.129993e+10,N/A


## Steering Loop — Top-5 Features

For each of the top-5 density-weighted features:
1. Build steering vector: `coeff × SAE.w_dec[feature_idx]` where `coeff = −10 × raw_max_activation`.
2. Run steered generation.
3. Extract the predicted label from the steered response.
4. Record results.

In [21]:
TOP_K_STEER = 5
prompt_ids  = full_ids[:, :prompt_len]   # (1, prompt_len) — prompt tokens only
gold_label  = sample_row["gold_label"].lower()

steering_results = []

for rank, row in feature_df.head(TOP_K_STEER).iterrows():
    fi      = int(row["feature_idx"])
    raw_max = row["max_activation"]
    coeff   = -raw_max              # negative = suppress

    steering_delta = (coeff * sae.w_dec[fi].float()).to(device=device)

    print(f"\n{'─'*70}")
    print(f"Rank {rank + 1:2d} | Feature {fi:>6d} | raw_max={raw_max:.3f} | coeff={coeff:.3f}")
    print(f"         | frac_nonzero={row['frac_nonzero']:.2e} | scaled_max={row['scaled_max_activation']:.3f}")
    print(f"         | Description: {row['description']}")

    steered_ids  = run_steered_generation(prompt_ids, LAYER, steering_delta)
    steered_text = decode_response(steered_ids)
    pred_label   = esnli_dataset.parse_model_answer(steered_text).label
    label_correct = pred_label == gold_label

    print(f"\nSteered response:")
    print(steered_text[:600] + ("…" if len(steered_text) > 600 else ""))
    print(f"\nPredicted label: {pred_label!r}  |  Gold: {gold_label!r}  |  Correct: {label_correct}")

    steering_results.append({
        "rank":             rank + 1,
        "feature_idx":      fi,
        "steering_value":   coeff,
        "steered_response": steered_text,
        "predicted_label":  pred_label,
        "correct_label":    label_correct,
    })

print(f"\n{'═'*70}")
print("Steering complete.")


──────────────────────────────────────────────────────────────────────
Rank  1 | Feature   1686 | raw_max=24749.104 | coeff=-24749.104
         | frac_nonzero=0.00e+00 | scaled_max=24749103515.625
         | Description: N/A

Steered response:
user Task: Determine the logical relationship between a Premise and a Hypothesis.
Options: entailment, contradiction, neutral.

Rules:
1. You MUST provide your reasoning inside <reasoning> tags.
2. You MUST provide the final label inside <label> tags.
3. The reasoning must come BEFORE the label.

Premise: A lady sitting on a bench that is against a building and under a poster of a man in a uniform waving.
Hypothesis: Nobody is sitting

model  برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار برقرار…

Predicted label: 'neutral'  |  Gold: 'contradiction'  |  Correct: False

──────────────────────────────────────────────────────────────────────
Rank  2 

## Results Summary

In [19]:
results_df = pd.DataFrame(steering_results)[
    ["rank", "feature_idx", "steering_value", "predicted_label", "correct_label"]
]
print(f"Gold label: '{gold_label}'")
print(f"Steerings that preserved the correct label: {results_df['correct_label'].sum()} / {len(results_df)}")
display(results_df)

Gold label: 'contradiction'
Steerings that preserved the correct label: 0 / 5


,rank,feature_idx,steering_value,predicted_label,correct_label
0,1,1686,-247491.035156,neutral,False
1,2,1224,-229845.546875,neutral,False
2,3,525,-227780.839844,neutral,False
3,4,1107,-225018.378906,neutral,False
4,5,410,-224842.636719,neutral,False
